# Multi-channel ξ SPLM — Full Training (8k / 16k steps)

## Motivation

The original α-init sweep ran **pilot** (4000 steps) on 8 α-initialisation
strategies. The best pilot configuration (`learned_from_uniform`,
α_init = [0.25, 0.50, 0.75, 0.95]) reached **14.69 PPL** at 4000 steps.

This notebook re-runs the winner at:
- **8,000 steps** (standard scaleup schedule, matching E9).
- **16,000 steps** (extended schedule, matching the Fock v2.1 ablation).

Goal: determine the converged Multi-Xi SPLM PPL for the HuggingFace
model card and for fair comparison with Fock v2.1 (9.30 PPL, 16k) and
Fock Attention (9.42 PPL, 16k).

## Model

`ScalarPotentialLMSARFMassLNMultiXi` (em_ln + K-EMA multi-ξ, causal-force)

| Parameter | Value |
|-----------|-------|
| d | 256 |
| L | 8 |
| v_hidden | 1024 |
| v_depth | 3 |
| K (ξ channels) | 4 |
| α_init | [0.25, 0.50, 0.75, 0.95] |
| α learnable | True |
| fixed_gamma | 0.30 |
| causal_force | True (leak-free) |
| params | ~16.5M |

## Baselines

| Model | PPL | Steps |
|-------|-----|-------|
| Multi-ξ SPLM pilot (α-sweep winner) | 14.69 | 4,000 |
| Multi-ξ PARFLM (comp K=8, H=128) | 12.06 | 8,000 |
| Fock v2.1 (τ_k + K_k + ortho) | 9.30 | 16,000 |
| Fock Attention (4-head) | 9.42 | 16,000 |
| Attention baseline | 7.81 | 8,000 |

## Arms

| # | Arm | Steps | Description |
|---|-----|-------|-------------|
| 1 | `scaleup_8k` | 8,000 | Standard E9 schedule |
| 2 | `extended_16k` | 16,000 | Extended schedule (2× warmup, 2× eval) |

## Hardware

- **A100 40GB**: ~1.5 h for 8k, ~3 h for 16k
- **L4 / T4**: ~4 h for 8k, ~8 h for 16k
- No gradient accumulation needed
- TF32 disabled is not required (no autograd.grad), but kept for reproducibility

## 1. Environment setup

In [ ]:
import os, sys, subprocess, shutil, json, time, math
from pathlib import Path

os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')

IN_COLAB = 'google.colab' in sys.modules
print('In Colab:', IN_COLAB)

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/semsimula_splm_multixi_rerun')
    REPO_PARENT = Path('/content')
else:
    DRIVE_ROOT = Path.home() / 'semsimula_splm_multixi_rerun'
    REPO_PARENT = Path.cwd().parent.parent.parent.parent

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS = DRIVE_ROOT / 'results'
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
print('Drive root   :', DRIVE_ROOT)
print('Results dir  :', DRIVE_RESULTS)

In [ ]:
REPO_URL = 'https://github.com/dimitarpg13/semsimula-paper.git'
REPO_DIR = REPO_PARENT / 'semsimula-paper'

if IN_COLAB:
    if REPO_DIR.exists():
        print(f'Repo already cloned at {REPO_DIR}')
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'],
                       check=False)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL,
                        str(REPO_DIR)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'torch', 'numpy', 'matplotlib', 'tiktoken', 'datasets'],
                   check=True)

SCRIPTS_DIR = REPO_DIR / 'notebooks' / 'conservative_arch' / 'scaleup'
MULTIXI_DIR = REPO_DIR / 'notebooks' / 'conservative_arch' / 'multixi'
assert SCRIPTS_DIR.exists(), f'Missing: {SCRIPTS_DIR}'
print('Scripts dir  :', SCRIPTS_DIR)

## 2. GPU check

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')
    DEVICE = 'cuda'
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    print('TF32 disabled for reproducibility')
elif torch.backends.mps.is_available():
    print('GPU: Apple MPS')
    DEVICE = 'mps'
else:
    print('WARNING: No GPU detected')
    DEVICE = 'cpu'

print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

## 3. Experiment configuration

Two arms using the winning α-init from the pilot sweep:
- `scaleup_8k`: standard 8,000-step E9 schedule
- `extended_16k`: 16,000-step extended schedule

In [ ]:
FIXED_GAMMA   = 0.30
SEED          = 0
MAX_TRAIN_TOK = 5_000_000

ARM_DEFS = {
    'scaleup_8k': {
        'mode': 'scaleup',
        'max_steps': 8000,
        'xi_channels': 4,
        'xi_alpha_inits': '0.25,0.50,0.75,0.95',
        'xi_alpha_init_mode': 'explicit',
        'desc': 'Standard 8k-step scaleup (E9 schedule) with best pilot α-init',
    },
    'extended_16k': {
        'mode': 'scaleup',
        'max_steps': 16000,
        'xi_channels': 4,
        'xi_alpha_inits': '0.25,0.50,0.75,0.95',
        'xi_alpha_init_mode': 'explicit',
        'desc': 'Extended 16k-step schedule (matching Fock v2.1 ablation)',
    },
}

ALL_ARMS = list(ARM_DEFS.keys())

print(f'Arms defined : {len(ALL_ARMS)}')
for name in ALL_ARMS:
    d = ARM_DEFS[name]
    print(f'  {name:20s}  {d["max_steps"]:5d} steps  K={d["xi_channels"]}  '
          f'α=[{d["xi_alpha_inits"]}]')
    print(f'  {"":20s}  {d["desc"]}')

In [ ]:
# ── SELECT ARMS TO RUN ────────────────────────────────────────────────────
# Edit ARMS_TO_RUN to target any subset.
#
# Examples:
#   ARMS_TO_RUN = ALL_ARMS                # both arms (default)
#   ARMS_TO_RUN = ['scaleup_8k']          # 8k only
#   ARMS_TO_RUN = ['extended_16k']        # 16k only
# ──────────────────────────────────────────────────────────────────────────

if 'results' not in dir():
    results = {}

ARMS_TO_RUN = ALL_ARMS

invalid = [n for n in ARMS_TO_RUN if n not in ARM_DEFS]
if invalid:
    raise ValueError(f'Unknown arm names: {invalid}\nValid: {ALL_ARMS}')

print(f'Selected {len(ARMS_TO_RUN)} / {len(ALL_ARMS)} arms:')
for name in ARMS_TO_RUN:
    d = ARM_DEFS[name]
    print(f'  {name:20s}  {d["max_steps"]} steps  —  {d["desc"]}')

## 4. Precompute logfreq surprisal (if needed)

In [ ]:
LOGFREQ_PATH = SCRIPTS_DIR / 'results' / 'logfreq_surprisal_tinystories.npy'

if not LOGFREQ_PATH.exists():
    print('Computing logfreq surprisal (one-time, ~2 min)...')
    subprocess.run(
        [sys.executable, str(SCRIPTS_DIR / 'compute_unigram_frequencies_tinystories.py')],
        cwd=str(SCRIPTS_DIR), check=True,
    )
    assert LOGFREQ_PATH.exists()
    print('Done.')
else:
    print(f'logfreq file exists: {LOGFREQ_PATH}')

## 5. Train Multi-ξ SPLM (8k / 16k)

Each arm trains `ScalarPotentialLMSARFMassLNMultiXi` with:
- K=4 learnable ξ-channels, α_init = [0.25, 0.50, 0.75, 0.95]
- causal_force = True (leak-free integrator)
- fixed_gamma = 0.30

Completed arms are skipped on re-run.  Progress is shown live below this cell.

In [ ]:
import re
from IPython.display import clear_output, display, HTML

# ── Log-line parsers (match [multixi-splm] prefixed stdout lines) ─────────
_TRAIN_RE = re.compile(
    r'step\s+(\d+)/(\d+)\s+'
    r'train\s+([\d.]+)\s+'
    r'lr\s+([\d.eE+-]+)\s+'
    r'grad\s+([\d.]+)\s+'
    r'm\[mean\s+([\d.]+)\s+std\s+([\d.]+)\]\s+'
    r'gamma=([\d.]+)\s+'
    r'\u03b1=\[([^\]]+)\]\s+'
    r'elapsed\s+([\d.]+)s'
)
_TRAIN_RE_ASCII = re.compile(
    r'step\s+(\d+)/(\d+)\s+'
    r'train\s+([\d.]+)\s+'
    r'lr\s+([\d.eE+-]+)\s+'
    r'grad\s+([\d.]+)\s+'
    r'm\[mean\s+([\d.]+)\s+std\s+([\d.]+)\]\s+'
    r'gamma=([\d.]+)\s+'
    r'.*?\[([\d.,]+)\]\s+'
    r'elapsed\s+([\d.]+)s'
)
_EVAL_RE = re.compile(
    r'step\s+(\d+)\s+val_loss=([\d.]+)\s+val_ppl=([\d.]+)'
)


def _parse_train(line):
    m = _TRAIN_RE.search(line)
    if not m:
        m = _TRAIN_RE_ASCII.search(line)
    if not m:
        return None
    return {
        'step': int(m.group(1)), 'total': int(m.group(2)),
        'train_loss': float(m.group(3)), 'lr': float(m.group(4)),
        'grad': float(m.group(5)),
        'mass_mean': float(m.group(6)), 'mass_std': float(m.group(7)),
        'gamma': float(m.group(8)),
        'alphas': m.group(9),
        'elapsed_s': float(m.group(10)),
    }


def _parse_eval(line):
    m = _EVAL_RE.search(line)
    if not m:
        return None
    return {'step': int(m.group(1)), 'val_loss': float(m.group(2)),
            'val_ppl': float(m.group(3))}


def _show_progress(arm_name, arm_def, tr, ev, best_ppl,
                   t_start, t_train_start, recent_lines, done=False):
    clear_output(wait=True)
    sys.stdout.flush()
    status = '\u2713 DONE' if done else '\u2699 RUNNING'
    W = 65
    print('\u2550' * W)
    print(f'  {status}  {arm_name}  ({arm_def["max_steps"]} steps)')
    print(f'  {arm_def["desc"]}')
    print('\u2550' * W)

    if tr:
        step, total = tr['step'], tr['total']
        elapsed = time.time() - t_start
        train_elapsed = (time.time() - t_train_start) if t_train_start else elapsed
        pct = 100 * step / total
        eta_s = train_elapsed * (total - step) / step if step > 0 else 0
        eta_str = 'Done!' if done else f'ETA {eta_s / 60:.1f} min'
        bar_w = 52
        filled = int(bar_w * pct / 100)
        bar = '\u2588' * filled + '\u2591' * (bar_w - filled)
        print(f'  [{bar}]')
        print(f'  Step : {step:5d} / {total}  ({pct:.1f}%)'
              f'   Elapsed: {elapsed / 60:.1f} min   {eta_str}')
        print()
        print(f'  train loss : {tr["train_loss"]:.4f}'
              f'   lr : {tr["lr"]:.2e}'
              f'   grad norm : {tr["grad"]:.3f}')
        print(f'  \u03b3          : {tr["gamma"]:.4f}'
              f'   mass : mean={tr["mass_mean"]:.3f}  std={tr["mass_std"]:.3f}')
        print(f'  \u03b1 channels : [{tr["alphas"]}]')

    if ev:
        print()
        best_str = f'  (best so far: {best_ppl:.3f})' if best_ppl < 1e6 else ''
        print(f'  Last val PPL : {ev["val_ppl"]:.3f}'
              f'   val loss : {ev["val_loss"]:.4f}'
              f'   (@ step {ev["step"]}){best_str}')

    noise = [l for l in recent_lines
             if l.strip()
             and '[multixi-splm] step' not in l
             and not l.startswith('{')]
    if noise:
        print()
        print('  Recent output:')
        for l in noise[-6:]:
            print(f'    {l}')


def _run_arm_streaming(arm_name, arm_def, cmd):
    """Launch trainer, stream stdout, show live progress."""
    all_lines, recent_lines = [], []
    tr, ev = None, None
    best_ppl = float('inf')
    t_start = time.time()
    t_train_start = None

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
        cwd=str(SCRIPTS_DIR),
        env={**os.environ, 'PYTHONUNBUFFERED': '1'},
    )
    try:
        for raw in proc.stdout:
            line = raw.rstrip()
            all_lines.append(line)
            recent_lines = all_lines[-40:]

            new_tr = _parse_train(line)
            new_ev = _parse_eval(line)
            if new_tr:
                tr = new_tr
                if t_train_start is None:
                    t_train_start = time.time()
            if new_ev:
                ev = new_ev
                if ev['val_ppl'] < best_ppl:
                    best_ppl = ev['val_ppl']
            if new_tr or new_ev:
                _show_progress(arm_name, arm_def, tr, ev, best_ppl,
                               t_start, t_train_start, recent_lines)
            elif tr is None:
                if line.strip() and not line.startswith('{'):
                    print(line, flush=True)
    except Exception as exc:
        print(f'\n  STREAM ERROR: {exc}')
    finally:
        proc.wait()

    _show_progress(arm_name, arm_def, tr, ev, best_ppl,
                   t_start, t_train_start, recent_lines, done=True)
    return proc.returncode, all_lines


# ── Training loop ─────────────────────────────────────────────────────────
TRAINER = str(SCRIPTS_DIR / 'train_splm_em_ln_multixi_scaleup.py')

for arm_name in ARMS_TO_RUN:
    arm_def = ARM_DEFS[arm_name]
    arm_results_dir = DRIVE_RESULTS / arm_name
    arm_results_dir.mkdir(parents=True, exist_ok=True)

    # Skip if summary file already exists (completed on a previous run)
    summary_glob = list(arm_results_dir.glob('*_summary.md'))
    if summary_glob:
        print(f'\n\u23ed  {arm_name}: SKIP (already complete)')
        with open(summary_glob[0]) as f:
            for line in f:
                if 'Final' in line or 'ppl' in line.lower():
                    print(f'   {line.strip()}')
        results[arm_name] = {'status': 'skipped'}
        continue

    cmd = [
        sys.executable, '-u', TRAINER,
        '--mode', arm_def['mode'],
        '--seed', str(SEED),
        '--fixed-gamma', str(FIXED_GAMMA),
        '--max-train-tokens', str(MAX_TRAIN_TOK),
        '--results-dir', str(arm_results_dir),
        '--tag-suffix', arm_name,
        '--logfreq-path', str(LOGFREQ_PATH),
        '--device', DEVICE,
        '--xi-channels', str(arm_def['xi_channels']),
        '--xi-alpha-init-mode', arm_def['xi_alpha_init_mode'],
        '--xi-alpha-inits', arm_def['xi_alpha_inits'],
        '--max-steps', str(arm_def['max_steps']),
    ]

    rc, all_lines = _run_arm_streaming(arm_name, arm_def, cmd)

    if rc != 0:
        print(f'\n  \u2717 FAILED: trainer exited with code {rc}')
        print('  Last 30 lines of output:')
        for l in all_lines[-30:]:
            print(f'    {l}')
        results[arm_name] = {'status': 'failed', 'returncode': rc}
        continue

    summary_files = list(arm_results_dir.glob('*_summary.md'))
    if summary_files:
        print('\n  \u2500\u2500 Training summary \u2500\u2500')
        with open(summary_files[0]) as f:
            print(f.read())

    results[arm_name] = {'status': 'completed'}

# Final status table
print(f'\n{"\u2550" * 65}')
print(f'All selected arms finished  ({len(ARMS_TO_RUN)} selected)')
for name, r in results.items():
    sym = {'completed': '\u2713', 'skipped': '\u23ed', 'failed': '\u2717'}.get(r['status'], '?')
    print(f'  {sym}  {name:20s}  {r["status"]}')

## 6. Results comparison

Compare 8k/16k runs against pilot baseline and other SPLM family members.

In [ ]:
import matplotlib.pyplot as plt

BASELINES = {
    'Multi-\u03be SPLM pilot (4k)': 14.69,
    'Multi-\u03be PARFLM (8k)': 12.06,
    'Fock v2.1 (16k)': 9.30,
    'Fock Attention (16k)': 9.42,
    'Hybrid SPLM+Attn (16k)': 8.01,
    'Attention baseline (8k)': 7.81,
}

arm_ppls = {}
arm_alphas = {}
arm_details = {}

for arm_name in ARM_DEFS:
    arm_dir = DRIVE_RESULTS / arm_name
    ckpt_files = list(arm_dir.glob('*_ckpt_latest.pt'))
    if not ckpt_files:
        continue
    ckpt = torch.load(ckpt_files[0], map_location='cpu', weights_only=False)
    arm_ppls[arm_name] = ckpt.get('final_val_ppl')
    arm_alphas[arm_name] = ckpt.get('final_xi_alphas')
    arm_details[arm_name] = {
        'elapsed_sec': ckpt.get('elapsed_sec'),
        'final_gamma': ckpt.get('final_gamma'),
        'steps': ckpt.get('train_cfg', {}).get('steps'),
    }

if not arm_ppls:
    print('No results found yet.')
else:
    print(f'{"Arm":20s} {"Steps":>6s} {"Final \u03b1":40s} {"PPL":>8s} {"\u0394 vs pilot":>10s}')
    print('\u2500' * 90)
    pilot_ppl = 14.69
    for name in sorted(arm_ppls, key=lambda x: arm_ppls[x]):
        alpha_str = ', '.join(f'{a:.4f}' for a in arm_alphas[name]) if arm_alphas[name] else '?'
        d = ARM_DEFS[name]
        delta = arm_ppls[name] - pilot_ppl
        steps = arm_details[name].get('steps', d['max_steps'])
        print(f'{name:20s} {steps:6d} [{alpha_str:38s}] {arm_ppls[name]:8.2f} {delta:+10.2f}')
    print('\u2500' * 90)
    for bname, bppl in BASELINES.items():
        print(f'{bname:20s} {"":>6s} {"":>40s} {bppl:8.2f}  (baseline)')

    best_arm = min(arm_ppls, key=lambda x: arm_ppls[x])
    best_ppl = arm_ppls[best_arm]
    print(f'\nBest: {best_arm} \u2192 {best_ppl:.2f} PPL'
          f'  (\u0394={best_ppl - pilot_ppl:+.2f} vs pilot)')

In [ ]:
if arm_ppls:
    # ── PPL bar chart ────────────────────────────────────────────────────────
    all_names = (list(sorted(arm_ppls, key=lambda x: arm_ppls[x]))
                 + list(BASELINES.keys()))
    all_ppls  = ([arm_ppls[n] for n in sorted(arm_ppls, key=lambda x: arm_ppls[x])]
                 + list(BASELINES.values()))
    colors = (['#2ecc71'] * len(arm_ppls)
              + ['#95a5a6'] * len(BASELINES))

    fig, ax = plt.subplots(figsize=(12, max(5, len(all_names) * 0.5)))
    bars = ax.barh(all_names, all_ppls, color=colors, edgecolor='white')
    for bar, ppl in zip(bars, all_ppls):
        ax.text(bar.get_width() + 0.15, bar.get_y() + bar.get_height()/2,
                f'{ppl:.2f}', va='center', fontsize=9)

    ax.axvline(x=14.69, color='#3498db', linestyle='--', linewidth=1.5,
               label='Pilot baseline (14.69)')
    ax.set_xlabel('Val PPL (lower is better)')
    ax.set_title('Multi-\u03be SPLM — Full Training Rerun (8k / 16k steps)')
    ax.invert_yaxis()
    ax.grid(True, axis='x', alpha=0.3)
    ax.legend(loc='lower right')
    fig.tight_layout()
    fig.savefig(DRIVE_RESULTS / 'splm_multixi_rerun_comparison.png', dpi=150)
    plt.show()

    # ── Convergence curves ────────────────────────────────────────────────────
    fig2, ax2 = plt.subplots(figsize=(10, 5))
    for arm_name in sorted(arm_ppls, key=lambda x: arm_ppls[x]):
        arm_dir = DRIVE_RESULTS / arm_name
        log_files = list(arm_dir.glob('*_training_log.jsonl'))
        if not log_files:
            continue
        steps_v, val_ppls = [], []
        with open(log_files[0]) as f:
            for line in f:
                rec = json.loads(line)
                if 'val_ppl' in rec:
                    steps_v.append(rec['step'])
                    val_ppls.append(rec['val_ppl'])
                elif 'val_loss' in rec and rec.get('val_loss'):
                    steps_v.append(rec['step'])
                    val_ppls.append(math.exp(rec['val_loss']))
        if steps_v:
            ax2.plot(steps_v, val_ppls, marker='o', markersize=3,
                     label=f'{arm_name} ({arm_ppls[arm_name]:.2f})')

    ax2.axhline(y=14.69, color='#3498db', linestyle='--', linewidth=1.5,
                label='Pilot baseline (14.69)')
    ax2.axhline(y=12.06, color='#e67e22', linestyle=':', linewidth=1.5,
                label='Multi-\u03be PARFLM (12.06)')
    ax2.axhline(y=9.30, color='#27ae60', linestyle=':', linewidth=1.5,
                label='Fock v2.1 (9.30)')
    ax2.set_xlabel('Step')
    ax2.set_ylabel('Val PPL')
    ax2.set_title('Multi-\u03be SPLM convergence (8k vs 16k)')
    ax2.grid(True, alpha=0.3)
    ax2.legend(fontsize=8, loc='upper right')
    fig2.tight_layout()
    fig2.savefig(DRIVE_RESULTS / 'splm_multixi_rerun_convergence.png', dpi=150)
    plt.show()

    # ── α evolution ───────────────────────────────────────────────────────────
    fig3, axes3 = plt.subplots(1, len(arm_ppls), figsize=(6 * len(arm_ppls), 4),
                                squeeze=False)
    for idx, arm_name in enumerate(sorted(arm_ppls, key=lambda x: ARM_DEFS[x]['max_steps'])):
        ax3 = axes3[0][idx]
        arm_dir = DRIVE_RESULTS / arm_name
        log_files = list(arm_dir.glob('*_training_log.jsonl'))
        if not log_files:
            continue
        steps_a = []
        alpha_tracks = {}
        with open(log_files[0]) as f:
            for line in f:
                rec = json.loads(line)
                if 'xi_alphas' in rec:
                    steps_a.append(rec['step'])
                    for k, a in enumerate(rec['xi_alphas']):
                        alpha_tracks.setdefault(k, []).append(a)
        for k in sorted(alpha_tracks):
            ax3.plot(steps_a, alpha_tracks[k], label=f'\u03b1_{k}')
        ax3.set_xlabel('Step')
        ax3.set_ylabel('\u03b1')
        ax3.set_title(f'{arm_name} ({arm_ppls[arm_name]:.2f} PPL)')
        ax3.set_ylim(-0.05, 1.05)
        ax3.legend(fontsize=8)
        ax3.grid(True, alpha=0.3)
    fig3.tight_layout()
    fig3.savefig(DRIVE_RESULTS / 'splm_multixi_rerun_alpha_evolution.png', dpi=150)
    plt.show()

## 7. Save consolidated report

In [ ]:
report = {
    'experiment': 'splm_multixi_rerun_8k_16k',
    'motivation': 'Full training of best pilot α-init at 8k and 16k steps',
    'model': 'ScalarPotentialLMSARFMassLNMultiXi (em_ln + multi-\u03be)',
    'config': {
        'fixed_gamma': FIXED_GAMMA,
        'seed': SEED,
        'max_train_tokens': MAX_TRAIN_TOK,
        'd': 256, 'L': 8, 'v_hidden': 1024, 'v_depth': 3,
        'xi_channels': 4,
        'xi_alpha_inits': [0.25, 0.50, 0.75, 0.95],
        'xi_learnable': True,
        'causal_force': True,
    },
    'arms': {},
    'baselines': BASELINES,
}

for name in arm_ppls:
    report['arms'][name] = {
        'max_steps': ARM_DEFS[name]['max_steps'],
        'final_ppl': arm_ppls[name],
        'final_alphas': arm_alphas.get(name),
        'final_gamma': arm_details.get(name, {}).get('final_gamma'),
        'elapsed_sec': arm_details.get(name, {}).get('elapsed_sec'),
    }

report_path = DRIVE_RESULTS / 'splm_multixi_rerun_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print(f'Report saved: {report_path}')
if IN_COLAB:
    print('Results are persisted on Google Drive at:')
    print(f'  {DRIVE_ROOT}')
    print()
    print('To upload the best checkpoint to HuggingFace, copy the results')
    print('directory from GDrive and use the upload script from the repo.')